## Install Required Libraries

In [0]:
%pip install yfinance==0.2.66 fredapi==0.5.2
dbutils.library.restartPython()

## Stocks used:

| Symbol | Company | Symbol | Company | Symbol | Company | Symbol | Company | Symbol | Company |
|--------|---------|--------|---------|--------|---------|--------|---------|--------|---------|
| AAPL   | Apple   | MSFT   | Microsoft | AMZN   | Amazon   | NVDA   | NVIDIA   | GOOGL  | Alphabet Class A |
| META   | Meta Platforms | TSLA   | Tesla   | BRK-B  | Berkshire Hathaway Class B | UNH    | UnitedHealth Group | XOM    | Exxon Mobil |
| JNJ    | Johnson & Johnson | JPM    | JPMorgan Chase | V      | Visa     | PG     | Procter & Gamble | MA     | Mastercard |
| HD     | Home Depot | CVX    | Chevron   | MRK    | Merck    | ABBV   | AbbVie   | LLY    | Eli Lilly |
| PEP    | PepsiCo  | KO     | Coca-Cola | COST   | Costco Wholesale | AVGO   | Broadcom | WMT    | Walmart |
| MCD    | McDonald’s | CSCO   | Cisco Systems | ACN    | Accenture | TMO    | Thermo Fisher Scientific | ABT    | Abbott Laboratories |
| DHR    | Danaher  | LIN    | Linde    | NEE    | NextEra Energy | PM     | Philip Morris International | TXN    | Texas Instruments |
| RTX    | RTX Corporation | UNP    | Union Pacific | HON    | Honeywell | LOW    | Lowe’s   | AMGN   | Amgen |
| IBM    | IBM      | GS     | Goldman Sachs | CAT    | Caterpillar | BA     | Boeing   | SBUX   | Starbucks |
| INTC   | Intel    | AMD    | Advanced Micro Devices | QCOM   | Qualcomm | DE     | Deere & Company | ISRG   | Intuitive Surgical |

In [0]:
LANDING_BASE = "s3://trading-signals-landing"

OHLCV_LANDING = f"{LANDING_BASE}/ohlcv"
MACRO_LANDING = f"{LANDING_BASE}/macro"

# S&P 500 — we'll start with the top 50 stocks by market cap for faster development.
SYMBOLS = [
    "AAPL", "MSFT", "AMZN", "NVDA", "GOOGL", "META", "TSLA", "BRK-B", "UNH", "XOM",
    "JNJ", "JPM", "V", "PG", "MA", "HD", "CVX", "MRK", "ABBV", "LLY",
    "PEP", "KO", "COST", "AVGO", "WMT", "MCD", "CSCO", "ACN", "TMO", "ABT",
    "DHR", "LIN", "NEE", "PM", "TXN", "RTX", "UNP", "HON", "LOW", "AMGN",
    "IBM", "GS", "CAT", "BA", "SBUX", "INTC", "AMD", "QCOM", "DE", "ISRG"
]

# Date range — 5 years of historical data
DATA_START = "2020-01-01"
DATA_END = "2025-04-21"

# FRED API key —  https://fred.stlouisfed.org/docs/api/api_key.html
# In the databricks CLI, run:
# databricks secrets create-scope raghavan-trading-signals
# databricks secrets put-secret raghavan-trading-signals fred-api-key --string-value "YOUR_KEY"

FRED_API_KEY = dbutils.secrets.get(scope="raghavan-trading-signals", key="fred-api-key")

## Extract Stock Data (OHLCV) using yfinance (from Yahoo Finance)


**OHLCV** = **O**pen, **H**igh, **L**ow, **C**lose, **V**olume
| Column | Meaning |
|---|---|
| `date` | Trading day |
| `open` | Price when market opened (9:30 AM ET) |
| `high` | Highest price during the day |
| `low` | Lowest price during the day |
| `close` | Price when market closed (4:00 PM ET) |
| `adj_close` | Close price adjusted for **dividends and splits** (use this for returns/backtesting) |
| `volume` | Number of shares traded that day |
| `dividends` | Cash paid per share that day (usually 0) |
| `stock_splits` | Split ratio if a split occurred (e.g., 4 for a 4-for-1 split) |
| `symbol` | Ticker (e.g., AAPL) |
| `ingestion_timestamp` | When *you* pulled the data — useful for auditing/debugging |


In [0]:
import yfinance as yf
import pandas as pd
from datetime import datetime, UTC

print(f"Downloading OHLCV data for {len(SYMBOLS)} stocks from {DATA_START} to {DATA_END}...")

all_stock_data = []

for i, symbol in enumerate(SYMBOLS):
    try:
        ticker = yf.Ticker(symbol)
        df = ticker.history(start=DATA_START, end=DATA_END, auto_adjust=False)

        if df.empty:
            print(f"  SKIP {symbol}: no data returned")
            continue

        df = df.reset_index()
        df["symbol"] = symbol
        df["ingestion_timestamp"] = datetime.now(UTC)

        # Rename columns to lowercase with underscores (clean convention)
        df.columns = [
            c.lower().replace(" ", "_") for c in df.columns
        ]

        all_stock_data.append(df)

        if (i + 1) % 10 == 0:
            print(f"  Downloaded {i + 1}/{len(SYMBOLS)} stocks...")

    except Exception as e:
        print(f"  ERROR {symbol}: {e}")

# Combine all stocks into one DataFrame
ohlcv_pdf = pd.concat(all_stock_data, ignore_index=True)
print(f"\nTotal rows: {len(ohlcv_pdf):,}")
print(f"Date range: {ohlcv_pdf['date'].min()} to {ohlcv_pdf['date'].max()}")
print(f"Stocks: {ohlcv_pdf['symbol'].nunique()}")

In [0]:
# Convert Pandas DataFrame to Spark DataFrame for writing to cloud storage
ohlcv_spark = spark.createDataFrame(ohlcv_pdf)

display(ohlcv_spark)

## Save OHLCV to Landing Zone

In [0]:
# Generate a timestamped filename so each extraction is a separate file.
# This is important: Autoloader (next checkpoint) will detect each new file.
timestamp = datetime.now(UTC).strftime("%Y%m%d_%H%M%S")
output_path = f"{OHLCV_LANDING}/ohlcv_{timestamp}.parquet"

ohlcv_spark.write.mode("overwrite").parquet(output_path)
print(f"Saved {len(ohlcv_pdf):,} rows to {output_path}")

## Extract Macro Economic Data from FRED


## Uses of FRED
**FRED** = **F**ederal **R**eserve **E**conomic **D**ata. We are using it to **fetch macroeconomic context** alongside stock prices. Stock prices alone don't tell the full story as markets move based on interest rates, inflation, fear levels, and the broader economy. By pulling these indicators, your trading signal generator can answer questions like:
- "Is the market scared right now?" (VIX)
- "Is borrowing cheap or expensive?" (Fed Funds Rate, Treasury yields)
- "Is inflation hot?" (CPIAUCSL)
- "Is the economy healthy?" (UNRATE)
---
## The 7 Indicators Explained
| Series ID | Name | What It Means (Simply) | Why Traders Care |
|---|---|---|---|
| **VIXCLS** | VIX (Volatility Index) | The market's "fear gauge." Measures expected stock market swings over the next 30 days. | High VIX (>30) = panic/fear → stocks often falling. Low VIX (<15) = calm/complacency. |
| **DFF** | Fed Funds Rate | The interest rate banks charge each other overnight. Set by the Federal Reserve. | When Fed raises rates → borrowing gets expensive → stocks usually drop. Cuts → stocks rally. |
| **DGS10** | 10-Year Treasury Yield | Interest rate the US government pays to borrow money for 10 years. | The "risk-free rate." Higher yields make stocks less attractive vs. bonds. |
| **DGS2** | 2-Year Treasury Yield | Same as above, but for 2-year government bonds. | Reflects short-term rate expectations. |
| **DTWEXBGS** | USD Index | Strength of the US dollar vs. a basket of foreign currencies. | Strong dollar hurts US exporters (Apple, etc.) and emerging markets. |
| **CPIAUCSL** | CPI (Consumer Price Index) | Measures inflation — how much prices of everyday goods rose. | High inflation → Fed raises rates → bad for stocks. |
| **UNRATE** | Unemployment Rate | % of people looking for work but jobless. | Low unemployment = strong economy, but can also signal Fed tightening. |
---
**How to use this data (Example)**: \
10Y minus 2Y is the **yield curve spread**. When it goes negative ("inversion"), a recession often follows within 12–18 months.

In [0]:
from fredapi import Fred

fred = Fred(api_key=FRED_API_KEY)

# Key economic indicators for trading signals
FRED_SERIES = {
    "VIXCLS":    "vix",               # CBOE Volatility Index
    "DFF":       "fed_funds_rate",     # Federal Funds Effective Rate
    "DGS10":     "treasury_10y",       # 10-Year Treasury Yield
    "DGS2":      "treasury_2y",        # 2-Year Treasury Yield
    "DTWEXBGS":  "usd_index",          # Trade-Weighted US Dollar Index
    "CPIAUCSL":  "cpi",                # Consumer Price Index
    "UNRATE":    "unemployment_rate",  # Unemployment Rate
}

macro_frames = []

for series_id, friendly_name in FRED_SERIES.items():
    try:
        data = fred.get_series(series_id, observation_start=DATA_START, observation_end=DATA_END)
        df = data.reset_index()
        df.columns = ["date", "value"]
        df["indicator"] = friendly_name
        df["series_id"] = series_id
        df["ingestion_timestamp"] = datetime.now(UTC)
        macro_frames.append(df)
        print(f"  {friendly_name}: {len(df)} observations")
    except Exception as e:
        print(f"  ERROR {series_id}: {e}")

macro_pdf = pd.concat(macro_frames, ignore_index=True)
print(f"\nTotal macro rows: {len(macro_pdf):,}")

In [0]:
macro_spark = spark.createDataFrame(macro_pdf)

display(macro_spark)

## Save Macro Data to Landing Zone

In [0]:
timestamp = datetime.now(UTC).strftime("%Y%m%d_%H%M%S")
output_path = f"{MACRO_LANDING}/macro_{timestamp}.parquet"

macro_spark.write.mode("overwrite").parquet(output_path)

print(f"Saved {len(macro_pdf):,} rows to {output_path}")

## Quick Verification

**Note**: 
- spark.write.parquet(path) always writes a directory, not a single file. 
- Spark is distributed, so each executor task writes its own part-*.snappy.parquet file in parallel.

In [0]:
display(dbutils.fs.ls(f"{OHLCV_LANDING}/ohlcv_20260421_114022.parquet"))


In [0]:
display(dbutils.fs.ls(f"{MACRO_LANDING}/macro_20260421_114301.parquet"))